# 08.02 Tracing Agents — 복잡한 에이전트의 트레이스 구조

`01_quickstart`에서 한 번의 `create_agent` 실행을 UI에서 봤다면, 이제는 **LangGraph StateGraph · Deep Agents 서브에이전트 · 비동기 태스크**처럼 중첩 구조가 있는 에이전트가 어떻게 트레이스로 그려지는지를 본다.

## 학습 목표

- `Run` · `Trace` · `Project`의 관계를 이해한다 (run = span, trace = span tree)
- LangGraph 서브그래프가 부모 트레이스 안에서 네임스페이스 자식으로 표시되는 방식을 확인한다
- Deep Agents의 동기 서브에이전트와 비동기 서브에이전트(`async_tasks` 채널) 트레이스 차이를 구분한다
- `thread_id` / `session_id` / `conversation_id` 로 여러 실행을 세션 뷰에 묶는다
- `client.create_feedback(run_id, key, score)` 로 런에 평가 점수를 부착한다
- 태그·메타데이터 기반 `client.list_runs(filter=...)` 로 프로그램적 필터링을 한다
- 400일 보존 한계를 넘기기 위해 주요 트레이스를 **데이터셋으로 영구화**한다

## 사전 준비

`.env` 에 다음 세 줄이 있어야 한다 (`01_quickstart` 참고).

```dotenv
LANGSMITH_API_KEY=lsv2_pt_...
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=langsmith-tracing-agents
```

이 노트북은 Deep Agents의 동기/비동기 서브에이전트 구조까지 다루므로 `deepagents>=0.5.0` 가 필요하다.

In [ ]:
# !pip install -U langsmith langchain langgraph deepagents langchain-openai

from dotenv import load_dotenv
import os
load_dotenv(override=True)

assert os.environ.get("LANGSMITH_API_KEY"), "LANGSMITH_API_KEY 미설정"
assert os.environ.get("LANGSMITH_TRACING") == "true", "LANGSMITH_TRACING=true 필요"
os.environ.setdefault("LANGSMITH_PROJECT", "langsmith-tracing-agents")
print("프로젝트:", os.environ["LANGSMITH_PROJECT"])

## 8.02.1 Run · Trace · Project 개념

LangSmith의 데이터 계층은 네 개의 레벨로 쌓인다.

| 레벨 | 정의 | 예 |
|------|------|----|
| **Project** | 같은 애플리케이션의 트레이스를 모아두는 컨테이너 | `langsmith-tracing-agents` |
| **Trace** | 하나의 사용자 요청을 처리하는 동안 만들어진 run 트리 (최대 25,000 run / trace) | 에이전트 한 번 invoke |
| **Run** | 단일 span — LLM 호출, tool 호출, chain 노드 등 | `ChatOpenAI`, `get_weather` |
| **Thread** | `thread_id`/`session_id`/`conversation_id` 로 묶인 여러 trace — 멀티턴 대화 뷰 | 한 사용자의 세션 |

Run 하나에는 `parent_run_id`, `trace_id`, `start_time`, `end_time`, `inputs`, `outputs`, `total_tokens`, `total_cost` 등이 붙는다. **Trace는 같은 `trace_id` 를 공유하는 run들의 트리**일 뿐이다.

<!-- phase-c:embed -->
![프로젝트 Runs 리스트 (데이터 축적 후)](../assets/images/langsmith/02_tracing_agents/00_runs_populated_full.png)

*Runs 탭 — 2일치 실행이 쌓인 상태. Name/Input/Output/Error/Latency/Dataset/Tokens/Cost/Tags/Metadata 17개 컬럼 제공.*

![LangGraph subgraph trace tree](../assets/images/langsmith/02_tracing_agents/01_subgraph_tree_namespace.png)

*`Thread t_demo_0001` 의 Trace View. `PatchToolCallsMiddleware → model → ChatOpenAI → TodoListMiddleware` 체인이 한 chat-turn 내부에 네임스페이스로 구성된다. 우측 Attributes 탭에서 Tags/Metadata/Runtime 자동 캡쳐 확인.*

## 8.02.2 LangGraph StateGraph 트레이스 트리

LangGraph 그래프는 **그래프가 루트 run**, 각 노드가 자식 run, 서브그래프는 네임스페이스가 붙은 손자 run으로 보인다. 서브그래프의 노드 이름은 UI에서 `parent_node:child_node` 형식으로 표시된다.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

class State(TypedDict):
    topic: str
    outline: str
    draft: str

# --- 서브그래프: 초안 작성 ---
def write_outline(state: State) -> dict:
    out = llm.invoke(f"다음 주제의 개요를 3줄로: {state['topic']}").content
    return {"outline": out}

def write_draft(state: State) -> dict:
    out = llm.invoke(f"다음 개요를 문단으로 확장: {state['outline']}").content
    return {"draft": out}

writer = (
    StateGraph(State)
    .add_node("outline", write_outline)
    .add_node("draft", write_draft)
    .add_edge(START, "outline")
    .add_edge("outline", "draft")
    .add_edge("draft", END)
    .compile()
)

# --- 부모 그래프: 리서치 후 writer 서브그래프 호출 ---
def research(state: State) -> dict:
    return {"topic": f"[researched] {state['topic']}"}

parent = (
    StateGraph(State)
    .add_node("research", research)
    .add_node("writer", writer)      # 서브그래프를 노드로 삽입
    .add_edge(START, "research")
    .add_edge("research", "writer")
    .add_edge("writer", END)
    .compile()
)

result = parent.invoke(
    {"topic": "LangSmith로 관측 가능한 에이전트 만들기"},
    config={"run_name": "writer-pipeline", "tags": ["demo:subgraph"]},
)
print(result["draft"][:200], "...")

UI에서 `writer-pipeline` 트레이스를 열면 루트(`writer-pipeline`) 아래 `research`, `writer` 노드가 자식이고, `writer` 안에 `writer:outline`, `writer:draft` 손자 run이 네임스페이스와 함께 표시된다. **서브그래프 경로가 run 이름에 그대로 박히므로** `name contains writer:` 같은 필터를 쓸 수 있다.

## 8.02.3 Deep Agents 서브에이전트 트레이스 (동기 · 비동기)

Deep Agents의 서브에이전트는 부모 run 아래 독립된 자식 트리로 나타난다.

- **동기** (`SubAgent` dict): 부모가 블로킹되므로 단일 trace. `task` 툴 호출 run 아래에 서브에이전트의 LLM/tool 호출이 중첩된다.
- **비동기** (`AsyncSubAgent`): 별개의 Agent Protocol 서버에서 실행되므로 부모와 **다른 trace**로 기록된다. 부모 상태의 `async_tasks` 채널에 `task_id` 만 남고, 부모 trace에는 `start_async_task` / `check_async_task` 같은 관리 tool 호출만 보인다.

<!-- phase-c:embed -->
![Deep Agents subagent trace + user_thumbs 1.00 feedback](../assets/images/langsmith/02_tracing_agents/02_subagent_sync_trace.png)

*Turn 2 선택 상태. 트리에서 `tools → task → researcher` 서브에이전트 체인이 보이고, 우측 Feedback 탭에 `user_thumbs 1.00` 점수, AI 응답 옆 `task(description=..., subagent_type=researcher)` 툴 호출이 나타난다. Turn 2 hover 툴팁의 `cache read 5K / $0.0005` 는 Anthropic/OpenAI prompt caching 효과.*

![Thread Turn View: 대화 흐름](../assets/images/langsmith/02_tracing_agents/05_thread_detail_conversation.png)

*같은 thread 의 Turn View — 각 turn 의 Input/Output 을 대화 버블로 본다. Turn 2 AI output 에 `task call_Qks...` description 과 `subagent_type: researcher` 가 YAML 로 표시.*

In [ ]:
from deepagents import create_deep_agent
from langchain.tools import tool

@tool
def fake_search(query: str) -> str:
    """가짜 검색 도구 — 실제 네트워크 호출 없이 고정 응답을 반환한다."""
    return f"{query} 에 대한 요약: LangSmith는 LangChain의 공식 관측 플랫폼이다."

research_subagent = {
    "name": "researcher",
    "description": "한 주제에 대해 간단히 검색하고 3줄로 요약한다.",
    "system_prompt": "검색 결과를 3줄 이내로 한국어로 요약하세요.",
    "tools": [fake_search],
}

supervisor = create_deep_agent(
    model="openai:gpt-4.1-mini",
    system_prompt="리서치가 필요하면 researcher 에게 위임하고, 최종 답변을 한국어로 정리하세요.",
    subagents=[research_subagent],
)

out = supervisor.invoke(
    {"messages": [{"role": "user", "content": "LangSmith가 뭔지 한 문단으로 요약해 줘."}]},
    config={
        "run_name": "deepagent:sync-subagent",
        "tags": ["demo:deepagent", "mode:sync"],
    },
)
print(out["messages"][-1].content)

UI에서 `deepagent:sync-subagent` 를 열면 `task` 툴 호출이 보이고, 그 아래에 서브에이전트의 LLM + `fake_search` 호출이 중첩된다. 비동기의 경우는 아래 스니펫으로만 구조를 확인한다 — 실제 실행은 `langgraph dev` 서버가 필요하다.

```python
from deepagents import AsyncSubAgent
researcher = AsyncSubAgent(name="researcher", description="장시간 리서치", graph_id="researcher")
# 부모 trace: start_async_task 툴 호출만
# 자식 trace: researcher 그래프가 별개 trace (동일 thread_id 로 묶임)
# 상태 보존: async_tasks 채널은 compaction 을 거쳐도 살아남음
```

## 8.02.4 세션 뷰 — `thread_id` · `session_id` · `conversation_id`

여러 번의 invoke 를 **하나의 대화**로 묶으려면 `metadata` 에 세션 식별자를 넣는다. LangSmith 는 `thread_id`, `session_id`, `conversation_id` 중 하나라도 있으면 자동으로 Threads 뷰에 엮는다.

<!-- phase-c:embed -->
![Threads 탭](../assets/images/langsmith/02_tracing_agents/04_thread_view.png)

*Threads 탭 — 동일 `thread_id` 를 공유한 run 들이 하나의 대화 세션으로 묶인다. First Input / Last Output / turns / tokens / cost / P50·P99 Latency 컬럼 자동 집계.*

In [ ]:
session_config = {
    "run_name": "chat-turn",
    "metadata": {
        "thread_id": "t_demo_0001",   # 같은 값을 공유하면 Threads 뷰에 한 줄로 묶임
        "user_id": "u_00123",
        "app_version": "0.5.0",
    },
    "tags": ["env:dev", "feature:chat"],
}

for turn in ["안녕", "오늘 서울 날씨 어때?", "고마워"]:
    supervisor.invoke(
        {"messages": [{"role": "user", "content": turn}]},
        config=session_config,
    )
print("세 번의 turn이 thread_id=t_demo_0001 로 묶여 기록됨 — UI의 Threads 탭에서 확인")

## 8.02.5 런에 피드백 부착 — `client.create_feedback`

평가 점수·사용자 thumbs-up/down·내부 QA 리뷰 결과는 **Feedback** 으로 run 에 붙인다.

- `key`: 피드백 이름 (예: `"correctness"`, `"user_thumbs"`)
- `score`: 0~1 사이 실수 또는 임의 숫자
- `value`, `comment`: 선택

In [ ]:
from langsmith import Client

client = Client()

# 가장 최근 루트 run 하나를 가져와 피드백을 달아본다
latest = next(client.list_runs(
    project_name=os.environ["LANGSMITH_PROJECT"],
    is_root=True,
    limit=1,
))

client.create_feedback(
    run_id=latest.id,
    key="user_thumbs",
    score=1,                         # 1=up, 0=down
    comment="빠르고 정확했음",
)

print(f"feedback 부착 완료 → run {latest.id}")

## 8.02.6 태그·메타데이터 기반 필터 쿼리

UI 필터와 똑같은 표현식을 `client.list_runs(filter=...)` 로 코드에서 쓸 수 있다. 회귀 테스트·야간 배치·대시보드 피딩에 유용하다.

<!-- phase-c:embed -->
![Add filter 메뉴: 15개 필드](../assets/images/langsmith/02_tracing_agents/06_add_filter_menu.png)

*`Add filter` 클릭 시 나오는 필드 목록 — Input/Output/Run Name/Run Type/Latency/Status/Error Message/Tag/Metadata/Feedback/Run ID/Trace ID/Thread ID/Token Count/Cost + Advanced. Tag 와 Metadata 가 별도 필드로 존재해 `tags contains env:dev` 같은 조건 쿼리 가능.*

In [ ]:
# 1) 특정 태그가 붙은 루트 run만 가져오기
runs = list(client.list_runs(
    project_name=os.environ["LANGSMITH_PROJECT"],
    filter='and(eq(is_root, true), has(tags, "demo:deepagent"))',
    limit=5,
))
for r in runs:
    print(f"{r.start_time:%H:%M:%S}  {r.name:30s}  tags={r.tags}")

# 2) metadata.thread_id 가 달린 run만 조회
thread_runs = list(client.list_runs(
    project_name=os.environ["LANGSMITH_PROJECT"],
    filter='eq(metadata_key, "thread_id")',
    limit=20,
))
print(f"thread_id 가 달린 run 수: {len(thread_runs)}")

## 8.02.7 400일 보존 한계 → 데이터셋으로 영구화

SaaS LangSmith 는 **ingestion 시점부터 400일** 후 trace 가 삭제된다. 평가 회귀에 쓰고 싶은 중요한 실행은 **Dataset 으로 영구화**해야 한다. `03_datasets_and_evaluation.ipynb` 에서 자세히 다루지만, 여기선 패턴만 본다.

In [ ]:
# 트레이스 → dataset 영구화
# langsmith 0.7.x 부터 add_runs_to_dataset 가 제거됨. create_examples 로 변환해서 추가.
from langsmith import Client

client = Client()
DS_NAME = "agent-golden-traces"
try:
    ds = client.create_dataset(
        DS_NAME,
        description="영구 보존할 우수 트레이스",
    )
except Exception:
    ds = client.read_dataset(dataset_name=DS_NAME)

runs = list(client.list_runs(
    project_name=os.environ["LANGSMITH_PROJECT"],
    run_type="chain",
    limit=5,
))

examples = [
    {"inputs": r.inputs, "outputs": r.outputs, "metadata": {"source_run_id": str(r.id)}}
    for r in runs if r.outputs
]
if examples:
    client.create_examples(dataset_id=ds.id, examples=examples)
print(f"{len(examples)}개 example을 dataset '{ds.name}'에 추가했다.")

## 체크리스트

- [ ] LangGraph 서브그래프의 run 이름이 `parent:child` 네임스페이스로 표시되는지 UI에서 확인
- [ ] Deep Agents 동기 서브에이전트는 `task` 툴 아래 자식 트리로, 비동기는 별개 trace 로 기록된다는 점 이해
- [ ] `metadata.thread_id` 가 같은 실행들이 Threads 뷰에 한 줄로 묶이는지 확인
- [ ] `client.create_feedback(run_id, key, score)` 로 최근 run 에 피드백 부착
- [ ] `client.list_runs(filter=...)` 로 태그·메타데이터 기반 프로그램 조회
- [ ] 주요 trace 를 dataset 으로 옮겨 400일 한계 대응

## 다음

- `03_datasets_and_evaluation.ipynb` — 이 trace/dataset 을 가지고 code evaluator · LLM-as-judge · pairwise 평가를 붙이는 법

## 참고

- Observability 개념: https://docs.langchain.com/langsmith/observability-concepts
- Threads(세션) 뷰: https://docs.langchain.com/langsmith/threads
- Feedback API: https://docs.langchain.com/langsmith/attach-user-feedback
- Filter query 문법: https://docs.langchain.com/langsmith/filter-traces-in-application